# TD Colzette 2026

Dans ce TP, nous explorons la simulation de l'interception lumineuse dans un couvert de colza en culture pure ainsi qu'un couvert associé de colza-féverolle.

## A. Simulations de référence

Import des librairies nécessaires à l'éxécution du modèle Colzette et à l'interprétation de ses sorties.

In [ ]:
from openalea.colzette import Colzette, scenario_monocrop_brassica, scenario_intercrop_brassica_faba
import pandas as pd
import matplotlib.pyplot as plt
import os, sys
os.environ["PATH"] = os.pathsep.join(
    [os.path.join(sys.prefix, "Scripts"), os.path.join(sys.prefix, "Library", "bin"), os.environ["PATH"]])


Visualisation des paramètres du modèle, leur signification, leur valeur min et leur valeur max

In [ ]:
Colzette.parameters()

## A.1 Simulation Monoculture colza classique avec paramètres par défaut

On initialise le modèle à l'aide d'un scénario prédéfini, *scenario_monocrop_brassica*, que l'on viendra ensuite modifier dans la suite du TD.

In [ ]:
rapeseed_monocrop = Colzette(**scenario_monocrop_brassica)
rapeseed_monocrop.run()

L'absorption cumulée par plant de colza au cours du temps

In [ ]:
monocrop_outputs = rapeseed_monocrop.rapeseed_outputs
fig, ax = plt.subplots()
ax.plot(monocrop_outputs["DAS"], monocrop_outputs["absolute_energy_cum_plant"], label='default rapeseed')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Cumulative absorbed energy (J/plant)')
ax.legend()
ax.set_title('Default rapeseed monocrop simulation')

Absorption cumulée finale du colza (J/plante)

In [ ]:
rapeseed_monocrop.last_day_rapeseed("absolute_energy_cum_plant")

Visualisation du couvert en croissance (20s avant affichage)

In [ ]:
rapeseed_monocrop.animate_scene(variable="Eabs", as_html=True) # Daily absorption (J/m2) per leaf

## A.2. Simulation d'une association colza-féverole avec paramètres par défaut

In [ ]:
rapeseed_faba_intercrop = Colzette(**scenario_intercrop_brassica_faba)
rapeseed_faba_intercrop.run()

L'absorption cumulée par plant de colza et féverole au cours du temps

In [ ]:
intercropped_rapeseed_outputs = rapeseed_faba_intercrop.rapeseed_outputs
intercropped_faba_outputs = rapeseed_faba_intercrop.faba_outputs
fig, ax = plt.subplots()
ax.plot(intercropped_rapeseed_outputs["DAS"], intercropped_rapeseed_outputs["absolute_energy_cum_plant"], label='default rapeseed')
ax.plot(intercropped_faba_outputs["DAS"], intercropped_faba_outputs["absolute_energy_cum_plant"], label='default faba')
ax.set_xlabel('Time (days)')
ax.set_ylabel('Cumulative absorbed energy (J/plant)')
ax.legend()
ax.set_title('Default intercropping simulation')

Absorption finale du colza et de la féverole (J/plante)

In [ ]:
rapeseed = rapeseed_faba_intercrop.last_day_rapeseed("absolute_energy_cum_plant")
faba = rapeseed_faba_intercrop.last_day_faba("absolute_energy_cum_plant")
rapeseed, faba

Visualisation du couvert en croissance (25s avant affichage)

In [ ]:
rapeseed_faba_intercrop.animate_scene(variable="Eabs", as_html=True) # Daily absorption (J/m2) per leaf

## A.3 Changer une valeur de paramètre

Afin de changer une valeur d'un ou plusieurs paramètres, on passe leur valeur directement lors de l'initialisation de colzette et on répète les opérations précédentes. Ici avec l'exemple de la hauteur maximale atteinte par la féverolle:

In [ ]:
intercropping_parametrized = Colzette(L_height_legume=23.193454, **scenario_intercrop_brassica_faba)
intercropping_parametrized.run(day=108)
intercropping_parametrized.plot_scene(variable="Eabs", as_html=True)

# B. Comparaison de simulations

Réaliser des simulations comparatives en changeant des valeurs de paramètres choisis (max 4), en faisant varier leur valeur entre leur valeur min et max, puis en l'exprimant relativement aux sorties des deux simulations de référence; *rapeseed_monocrop* et *rapeseed_faba_intercrop*. Les valeurs min et max à tester sont regroupées dans Colzette.parameters().

Indiquez ci-dessous les paramètres choisis et vos hypothèses associées:
  -
  -
  -
  -
  -


Exemple d'excecution:

In [ ]:
parameters_variation = [
    # First comparision in monocrop
    dict(parameter="ins_angle_brassica", min=20., max=50., reference=rapeseed_monocrop, defaults=scenario_monocrop_brassica),
    # Second comparision in intercrop
    dict(parameter="ins_angle_brassica", min=20., max=50., reference=rapeseed_faba_intercrop, defaults=scenario_intercrop_brassica_faba),
    # ... add more comparision scenarios here
]

time_series = []
final_snapshots = []

for config in parameters_variation:
    parameter = config["parameter"]
    min = config['min']
    max = config['max']
    ref = config['reference']
    defaults = config['defaults']
    name = defaults['Type_simul']

    print(f"Starting {name} for {parameter} at min value {min}...")
    defaults[parameter] = min
    colzette_min = Colzette(**defaults)
    colzette_min.run()
    print("Done")

    print(f"Starting {name} for {parameter} at max value {max}...")
    defaults[parameter] = max
    colzette_max = Colzette(**defaults)
    colzette_max.run()
    print("Done")

    time_series.append((parameter, 
                        name, 
                        ref.rapeseed_outputs, 
                        colzette_min.rapeseed_outputs, 
                        colzette_max.rapeseed_outputs))
    final_snapshots.append((parameter, 
                        name, 
                        ref.last_day_rapeseed("absolute_energy_cum_plant"), 
                        colzette_min.last_day_rapeseed("absolute_energy_cum_plant"), 
                        colzette_max.last_day_rapeseed("absolute_energy_cum_plant")))



Enfin réaliser des graphes comparatifs (énergie absorbée en pourcentage relatif du scnénario de référence pour min et max), soit sur les séries temporelles, soit pour l'état final de la simulation:

In [ ]:
import numpy as np

# Final state: relative difference (%) of the min and max simulations to their reference
labels = [f"{parameter}\n{name}" for parameter, name, _, _, _ in final_snapshots]
rel_min = [100 * (v_min - ref) / ref for _, _, ref, v_min, _ in final_snapshots]
rel_max = [100 * (v_max - ref) / ref for _, _, ref, _, v_max in final_snapshots]

x = np.arange(len(labels))
fig, ax = plt.subplots()
ax.bar(x - 0.2, rel_min, width=0.4, label="min value")
ax.bar(x + 0.2, rel_max, width=0.4, label="max value")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x, labels)
ax.set_ylabel("Relative difference to reference (%)")
ax.set_title("Final cumulative absorbed energy per rapeseed plant")
ax.legend()

In [ ]:
# Time series: relative difference (%) of the min and max simulations to their reference, one plot per comparison
variable = "absolute_energy_cum_plant"

fig, axes = plt.subplots(len(time_series), 1, sharex=True, squeeze=False, figsize=(6, 3 * len(time_series)))
for ax, (parameter, name, ref, out_min, out_max) in zip(axes[:, 0], time_series):
    reference = ref.set_index("DAS")[variable]
    reference = reference.where(reference != 0)  # no relative difference before emergence
    for out, label in ((out_min, "min value"), (out_max, "max value")):
        rel = 100 * (out.set_index("DAS")[variable] - reference) / reference
        ax.plot(rel.index, rel.values, label=label)
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_title(f"{parameter}, {name}")
    ax.set_ylabel("Relative difference (%)")
    ax.legend()
axes[-1, 0].set_xlabel("Time (days)")
fig.tight_layout()